# Coral Video V2 — Stable Tracking + Larger Overlay

## YOLO11n Stage 3 → BoT-SORT → BBox Smoothing → MobileNetV4 V3 → Temporal Species Smoothing

This notebook fixes video flicker by adding:

- persistent **BoT-SORT Track IDs**
- **EMA bounding-box smoothing**
- **2-frame short-gap hold**
- motion-aware box prediction during the gap
- **temporal species probability smoothing**
- suppression of one-frame flashing tracks
- **larger text**
- **thicker bounding-box lines**

No training is performed.


In [ ]:
# ============================================================
# CELL 1 — INSTALL + MOUNT + IMPORTS
# ============================================================

!pip -q install ultralytics==8.4.117 timm==1.0.22

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import deque

import gc
import csv
import json
import math
import time
import shutil
import zipfile
import random

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import timm

from timm.data import resolve_model_data_config, create_transform
from ultralytics import YOLO

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU runtime in Colab.")

DEVICE = torch.device("cuda")
YOLO_DEVICE = 0

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print("=" * 115)
print("ENVIRONMENT")
print("=" * 115)
print("GPU  :", GPU_NAME)
print(f"VRAM : {VRAM_GB:.2f} GB")
print("Torch:", torch.__version__)
print("timm :", timm.__version__)
print("STATUS: ENVIRONMENT_READY")


In [ ]:
# ============================================================
# CELL 2 — PATHS + VIDEO ZIP
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Coral_Reef_AI_Project"
)

DETECTOR_CKPT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_Stage3_TargetedSliced_SmallCrowded"
    / "Detection_Stage3"
    / "weights"
    / "best.pt"
)

CLASSIFIER_CKPT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "MobileNetV4_Classification_V3_288_Context_HardMining"
    / "best_model_v3_macro_f1.pth"
)

EXPECTED_ZIP = Path(
    "/content/drive/MyDrive/Coral_video/coral_video.zip"
)

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

def find_coral_video_zip():
    if EXPECTED_ZIP.exists():
        return EXPECTED_ZIP

    exact = list(MYDRIVE.rglob("coral_video.zip"))
    if exact:
        return exact[0]

    candidates = [
        p
        for p in MYDRIVE.rglob("*.zip")
        if "coral" in p.name.lower()
        and "video" in p.name.lower()
    ]

    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        "Could not find coral_video.zip in MyDrive. "
        "If needed, add the shared folder as a shortcut to My Drive."
    )

VIDEO_ZIP = find_coral_video_zip()

for p in [DETECTOR_CKPT, CLASSIFIER_CKPT, VIDEO_ZIP]:
    if not p.exists():
        raise FileNotFoundError(p)

OUTPUT_DRIVE_ROOT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "Video_V2_BoTSORT_Stable_Stage3_MobileNetV4_V3"
)

OUTPUT_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 115)
print("PATH CHECK")
print("=" * 115)
print("Detector:", DETECTOR_CKPT)
print("Classifier:", CLASSIFIER_CKPT)
print("Video ZIP:", VIDEO_ZIP)
print("Output:", OUTPUT_DRIVE_ROOT)
print("STATUS: PATHS_READY")


In [ ]:
# ============================================================
# CELL 3 — EXTRACT ZIP + FIND VIDEOS
# ============================================================

WORK_ROOT = Path("/content/coral_video_v2")
INPUT_ROOT = WORK_ROOT / "input_videos"
LOCAL_OUTPUT_ROOT = WORK_ROOT / "outputs"

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

INPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(VIDEO_ZIP, "r") as zf:
    zf.extractall(INPUT_ROOT)

VIDEO_EXTENSIONS = {
    ".mp4", ".mov", ".avi", ".mkv",
    ".m4v", ".webm", ".mpg", ".mpeg",
}

video_paths = sorted([
    p
    for p in INPUT_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in VIDEO_EXTENSIONS
])

if not video_paths:
    raise RuntimeError("No supported videos found inside the ZIP.")

print("=" * 115)
print("VIDEOS FOUND")
print("=" * 115)

for i, p in enumerate(video_paths, start=1):
    print(f"{i:02d}. {p.relative_to(INPUT_ROOT)}")

print("Total videos:", len(video_paths))
print("STATUS: VIDEOS_DISCOVERED")


In [ ]:
# ============================================================
# CELL 4 — TRACKING / SMOOTHING / OVERLAY CONFIG
# ============================================================

DETECTOR_IMGSZ = 960

TRACK_INPUT_CONF = 0.08
DETECTOR_NMS_IOU = 0.70

TRACK_HIGH_THRESH = 0.25
TRACK_LOW_THRESH = 0.08
NEW_TRACK_THRESH = 0.35
TRACK_BUFFER = 30
TRACK_MATCH_THRESH = 0.80

MIN_TRACK_HITS_TO_DRAW = 2

BBOX_EMA_ALPHA = 0.45
VELOCITY_EMA_ALPHA = 0.50

MAX_MISSED_FRAMES = 2
STATE_MAX_AGE = 45

CLASSIFIER_INPUT = 288
CONTEXT_PAD = 0.10
CLASS_PROB_EMA_ALPHA = 0.25
CLASSIFIER_LOW_CONF = 0.50

# ------------------------------------------------------------
# BIGGER / THICKER OVERLAY AS REQUESTED
# ------------------------------------------------------------

BOX_THICKNESS = 5
HELD_BOX_THICKNESS = 4

FONT_SCALE = 0.85
TEXT_THICKNESS = 2

LABEL_PAD_X = 9
LABEL_PAD_Y = 7
LINE_GAP = 8

SHOW_TRACK_ID = True
DRAW_TRAIL = False
TRAIL_LENGTH = 15

FRAME_STRIDE = 1
MAX_FRAMES_PER_VIDEO = None
DRAW_HELD_TRACKS = True

TRACKER_YAML = WORK_ROOT / "coral_botsort_v2.yaml"

tracker_yaml_text = "\n".join([
    "tracker_type: botsort",
    f"track_high_thresh: {TRACK_HIGH_THRESH}",
    f"track_low_thresh: {TRACK_LOW_THRESH}",
    f"new_track_thresh: {NEW_TRACK_THRESH}",
    f"track_buffer: {TRACK_BUFFER}",
    f"match_thresh: {TRACK_MATCH_THRESH}",
    "fuse_score: True",
    "",
    "gmc_method: sparseOptFlow",
    "",
    "proximity_thresh: 0.5",
    "appearance_thresh: 0.8",
    "with_reid: False",
    "model: auto",
]) + "\n"

TRACKER_YAML.write_text(
    tracker_yaml_text,
    encoding="utf-8",
)

print("=" * 115)
print("VIDEO V2 CONFIG")
print("=" * 115)
print("Tracker             : BoT-SORT")
print("Input conf          :", TRACK_INPUT_CONF)
print("New track thresh    :", NEW_TRACK_THRESH)
print("Track buffer        :", TRACK_BUFFER)
print("Min hits to draw    :", MIN_TRACK_HITS_TO_DRAW)
print("BBox EMA alpha      :", BBOX_EMA_ALPHA)
print("Class EMA alpha     :", CLASS_PROB_EMA_ALPHA)
print("Gap hold            :", MAX_MISSED_FRAMES)
print("Box thickness       :", BOX_THICKNESS)
print("Font scale          :", FONT_SCALE)
print("Text thickness      :", TEXT_THICKNESS)
print("STATUS: VIDEO_V2_CONFIG_READY")


In [ ]:
# ============================================================
# CELL 5 — LOAD MOBILE V3 CLASSIFIER
# ============================================================

cls_ckpt = torch.load(
    CLASSIFIER_CKPT,
    map_location=DEVICE,
    weights_only=False,
)

FALLBACK_MODEL_NAME = (
    "mobilenetv4_conv_small_050.e3000_r224_in1k"
)

MODEL_NAME = cls_ckpt.get(
    "model_name",
    FALLBACK_MODEL_NAME,
)

DEFAULT_CLASSES = [
    "Euphflfiaancora",
    "Favosites",
    "Platygyra",
    "Sarcophyton",
    "Sinularia",
    "WavingHand",
]

class_to_idx = cls_ckpt.get(
    "class_to_idx",
    None,
)

if class_to_idx is None:
    class_to_idx = {
        c: i
        for i, c in enumerate(DEFAULT_CLASSES)
    }
else:
    class_to_idx = {
        str(k): int(v)
        for k, v in class_to_idx.items()
    }

idx_to_class = {
    int(v): str(k)
    for k, v in class_to_idx.items()
}

NUM_CLASSES = len(idx_to_class)

if NUM_CLASSES != 6:
    raise RuntimeError(
        f"Expected 6 classifier classes, got {NUM_CLASSES}"
    )

classifier = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=NUM_CLASSES,
).to(DEVICE)

classifier.load_state_dict(
    cls_ckpt["model_state_dict"]
)

classifier.eval()

data_cfg = resolve_model_data_config(
    classifier
)

data_cfg["input_size"] = (
    3,
    CLASSIFIER_INPUT,
    CLASSIFIER_INPUT,
)

if "mean" in cls_ckpt:
    data_cfg["mean"] = tuple(cls_ckpt["mean"])

if "std" in cls_ckpt:
    data_cfg["std"] = tuple(cls_ckpt["std"])

cls_transform = create_transform(
    **data_cfg,
    is_training=False,
)

print("=" * 115)
print("CLASSIFIER READY")
print("=" * 115)
print("Model   :", MODEL_NAME)
print("Input   :", CLASSIFIER_INPUT)
print("Context :", CONTEXT_PAD)
print("Classes :", class_to_idx)
print("STATUS: CLASSIFIER_READY")


In [ ]:
# ============================================================
# CELL 6 — HELPERS: CROP + CLASSIFICATION + TRACK STATE
# ============================================================

def padded_crop_from_xyxy(
    frame_bgr,
    box,
    padding=0.10,
):
    H, W = frame_bgr.shape[:2]

    x1, y1, x2, y2 = map(float, box)

    bw = max(0.0, x2 - x1)
    bh = max(0.0, y2 - y1)

    if bw <= 1 or bh <= 1:
        return None, None

    pad_x = bw * padding
    pad_y = bh * padding

    px1 = max(0, int(math.floor(x1 - pad_x)))
    py1 = max(0, int(math.floor(y1 - pad_y)))
    px2 = min(W, int(math.ceil(x2 + pad_x)))
    py2 = min(H, int(math.ceil(y2 + pad_y)))

    if px2 <= px1 or py2 <= py1:
        return None, None

    crop = frame_bgr[py1:py2, px1:px2]

    if crop.size == 0:
        return None, None

    return crop, (px1, py1, px2, py2)


def classify_crops(crops_bgr):
    if not crops_bgr:
        return []

    tensors = []

    for crop_bgr in crops_bgr:
        rgb = cv2.cvtColor(
            crop_bgr,
            cv2.COLOR_BGR2RGB,
        )

        pil_crop = Image.fromarray(rgb)

        tensors.append(
            cls_transform(pil_crop)
        )

    batch = torch.stack(
        tensors
    ).to(
        DEVICE,
        non_blocking=True,
    )

    with torch.no_grad():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = classifier(batch)

        probs = torch.softmax(
            logits.float(),
            dim=1,
        )

    probs_np = (
        probs
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    del batch
    del logits
    del probs

    return [p for p in probs_np]


def ema_box(
    previous_box,
    current_box,
    alpha=BBOX_EMA_ALPHA,
):
    current_box = np.asarray(
        current_box,
        dtype=np.float32,
    )

    if previous_box is None:
        return current_box.copy()

    previous_box = np.asarray(
        previous_box,
        dtype=np.float32,
    )

    return (
        alpha * current_box
        + (1.0 - alpha) * previous_box
    ).astype(np.float32)


def ema_probs(
    previous_probs,
    current_probs,
    alpha=CLASS_PROB_EMA_ALPHA,
):
    current_probs = np.asarray(
        current_probs,
        dtype=np.float32,
    )

    if previous_probs is None:
        smoothed = current_probs.copy()
    else:
        smoothed = (
            alpha * current_probs
            + (1.0 - alpha)
            * np.asarray(
                previous_probs,
                dtype=np.float32,
            )
        )

    total = float(smoothed.sum())

    if total > 0:
        smoothed = smoothed / total

    return smoothed.astype(np.float32)


def create_track_state(
    track_id,
    current_box,
    frame_idx,
    det_conf,
):
    current_box = np.asarray(
        current_box,
        dtype=np.float32,
    ).copy()

    return {
        "track_id": int(track_id),
        "smooth_box": current_box.copy(),
        "velocity": np.zeros(
            4,
            dtype=np.float32,
        ),
        "last_raw_box": current_box.copy(),
        "prob_ema": None,
        "species": None,
        "cls_conf": 0.0,
        "last_det_conf": float(det_conf),
        "last_seen_frame": int(frame_idx),
        "missed_frames": 0,
        "hits": 0,
        "trail": deque(maxlen=TRAIL_LENGTH),
        "last_raw_species": None,
        "last_stable_species": None,
        "raw_species_switches": 0,
        "stable_species_switches": 0,
    }


def update_smoothed_box(
    state,
    current_raw_box,
):
    old_smooth = np.asarray(
        state["smooth_box"],
        dtype=np.float32,
    ).copy()

    new_smooth = ema_box(
        old_smooth,
        current_raw_box,
        BBOX_EMA_ALPHA,
    )

    instantaneous_velocity = (
        new_smooth - old_smooth
    )

    state["velocity"] = (
        VELOCITY_EMA_ALPHA
        * instantaneous_velocity
        + (1.0 - VELOCITY_EMA_ALPHA)
        * state["velocity"]
    ).astype(np.float32)

    state["smooth_box"] = new_smooth


def predicted_hold_box(
    state,
    missed_frames,
):
    return (
        np.asarray(
            state["smooth_box"],
            dtype=np.float32,
        )
        + np.asarray(
            state["velocity"],
            dtype=np.float32,
        )
        * float(missed_frames)
    ).astype(np.float32)


def state_center(box):
    x1, y1, x2, y2 = map(float, box)

    return (
        int(round((x1 + x2) / 2.0)),
        int(round((y1 + y2) / 2.0)),
    )


print("STATUS: TRACK_HELPERS_READY")


In [ ]:
# ============================================================
# CELL 7 — LARGER DRAWING HELPERS
# ============================================================

def clip_box_to_frame(
    box,
    W,
    H,
):
    x1, y1, x2, y2 = map(float, box)

    x1 = max(0, min(x1, W - 1))
    y1 = max(0, min(y1, H - 1))
    x2 = max(0, min(x2, W - 1))
    y2 = max(0, min(y2, H - 1))

    return (
        int(round(x1)),
        int(round(y1)),
        int(round(x2)),
        int(round(y2)),
    )


def draw_track(
    frame,
    state,
    source="DETECTED",
    display_box=None,
):
    H, W = frame.shape[:2]

    if display_box is None:
        display_box = state["smooth_box"]

    x1, y1, x2, y2 = clip_box_to_frame(
        display_box,
        W,
        H,
    )

    species = (
        state["species"]
        if state["species"] is not None
        else "Unclassified"
    )

    cls_conf = float(
        state["cls_conf"]
    )

    det_conf = float(
        state["last_det_conf"]
    )

    track_id = int(
        state["track_id"]
    )

    if source == "HELD":
        color = (0, 200, 255)
        thickness = HELD_BOX_THICKNESS

    elif cls_conf < CLASSIFIER_LOW_CONF:
        color = (0, 165, 255)
        thickness = BOX_THICKNESS

    else:
        color = (0, 255, 0)
        thickness = BOX_THICKNESS

    cv2.rectangle(
        frame,
        (x1, y1),
        (x2, y2),
        color,
        thickness,
    )

    id_text = (
        f"ID {track_id} | "
        if SHOW_TRACK_ID
        else ""
    )

    hold_text = (
        f" | HOLD {state['missed_frames']}"
        if source == "HELD"
        else ""
    )

    line1 = (
        f"{id_text}{species}{hold_text}"
    )

    line2 = (
        f"Det {det_conf:.2f}"
        f" | Cls {cls_conf:.2f}"
    )

    font = cv2.FONT_HERSHEY_SIMPLEX

    (w1, h1), _ = cv2.getTextSize(
        line1,
        font,
        FONT_SCALE,
        TEXT_THICKNESS,
    )

    (w2, h2), _ = cv2.getTextSize(
        line2,
        font,
        FONT_SCALE,
        TEXT_THICKNESS,
    )

    label_w = (
        max(w1, w2)
        + 2 * LABEL_PAD_X
    )

    label_h = (
        h1
        + h2
        + 2 * LABEL_PAD_Y
        + LINE_GAP
    )

    label_x1 = x1

    if y1 - label_h >= 0:
        label_y1 = y1 - label_h
        label_y2 = y1
    else:
        label_y1 = y1
        label_y2 = min(
            H - 1,
            y1 + label_h
        )

    label_x2 = min(
        W - 1,
        label_x1 + label_w
    )

    cv2.rectangle(
        frame,
        (label_x1, label_y1),
        (label_x2, label_y2),
        color,
        -1,
    )

    text1_y = (
        label_y1
        + LABEL_PAD_Y
        + h1
    )

    text2_y = (
        text1_y
        + LINE_GAP
        + h2
    )

    cv2.putText(
        frame,
        line1,
        (
            label_x1 + LABEL_PAD_X,
            text1_y,
        ),
        font,
        FONT_SCALE,
        (0, 0, 0),
        TEXT_THICKNESS,
        cv2.LINE_AA,
    )

    cv2.putText(
        frame,
        line2,
        (
            label_x1 + LABEL_PAD_X,
            text2_y,
        ),
        font,
        FONT_SCALE,
        (0, 0, 0),
        TEXT_THICKNESS,
        cv2.LINE_AA,
    )

    if DRAW_TRAIL:
        pts = list(
            state["trail"]
        )

        if len(pts) >= 2:
            for a, b in zip(
                pts[:-1],
                pts[1:],
            ):
                cv2.line(
                    frame,
                    a,
                    b,
                    color,
                    2,
                    cv2.LINE_AA,
                )


print("STATUS: LARGE_OVERLAY_HELPERS_READY")


In [ ]:
# ============================================================
# CELL 8 — PROCESS ALL VIDEOS
# ============================================================

all_video_summaries = []


for video_number, video_path in enumerate(
    video_paths,
    start=1,
):
    print("\n" + "=" * 125)
    print(
        f"VIDEO {video_number}/{len(video_paths)}"
    )
    print("=" * 125)
    print("Input:", video_path.name)

    # Fresh detector instance for every video.
    video_detector = YOLO(
        str(DETECTOR_CKPT)
    )

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        print(
            "WARNING: cannot open. Skipping:",
            video_path
        )
        continue

    source_fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )

    if (
        not np.isfinite(source_fps)
        or source_fps <= 0
    ):
        source_fps = 25.0

    W = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    H = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    reported_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    output_fps = (
        source_fps / FRAME_STRIDE
    )

    safe_stem = (
        video_path.stem
        .replace(" ", "_")
    )

    local_dir = (
        LOCAL_OUTPUT_ROOT
        / safe_stem
    )

    local_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    local_video = (
        local_dir
        / f"{safe_stem}_V2_tracked_smoothed.mp4"
    )

    local_csv = (
        local_dir
        / f"{safe_stem}_V2_tracks.csv"
    )

    writer = cv2.VideoWriter(
        str(local_video),
        cv2.VideoWriter_fourcc(*"mp4v"),
        output_fps,
        (W, H),
    )

    if not writer.isOpened():
        cap.release()
        raise RuntimeError(
            f"Could not create output video: {local_video}"
        )

    csv_file = open(
        local_csv,
        "w",
        newline="",
        encoding="utf-8",
    )

    csv_writer = csv.DictWriter(
        csv_file,
        fieldnames=[
            "video",
            "source_frame",
            "output_frame",
            "time_sec",
            "track_id",
            "source",

            "raw_x1",
            "raw_y1",
            "raw_x2",
            "raw_y2",

            "display_x1",
            "display_y1",
            "display_x2",
            "display_y2",

            "det_conf",

            "raw_species",
            "raw_cls_conf",

            "stable_species",
            "stable_cls_conf",

            "missed_frames",
            "track_hits",

            "tracker_detector_ms",
            "classifier_ms",
            "frame_pipeline_ms",
        ],
    )

    csv_writer.writeheader()

    track_states = {}

    source_frame_idx = -1
    output_frame_idx = 0

    unique_ids = set()

    detected_track_rows = 0
    held_track_rows = 0

    tracker_times = []
    classifier_times = []
    pipeline_times = []

    video_start = time.perf_counter()


    while True:
        ok, frame = cap.read()

        if not ok:
            break

        source_frame_idx += 1

        if (
            source_frame_idx % FRAME_STRIDE
            != 0
        ):
            continue

        if (
            MAX_FRAMES_PER_VIDEO is not None
            and output_frame_idx >= MAX_FRAMES_PER_VIDEO
        ):
            break

        frame_start = time.perf_counter()

        # ----------------------------------------------------
        # 1) YOLO + BoT-SORT
        # ----------------------------------------------------

        tracker_start = time.perf_counter()

        result = video_detector.track(
            source=frame,
            persist=True,
            tracker=str(TRACKER_YAML),
            imgsz=DETECTOR_IMGSZ,
            conf=TRACK_INPUT_CONF,
            iou=DETECTOR_NMS_IOU,
            max_det=300,
            device=YOLO_DEVICE,
            verbose=False,
        )[0]

        tracker_ms = (
            time.perf_counter()
            - tracker_start
        ) * 1000.0

        current_tracks = []

        if (
            result.boxes is not None
            and len(result.boxes) > 0
            and result.boxes.id is not None
        ):
            raw_boxes = (
                result.boxes.xyxy
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            raw_confs = (
                result.boxes.conf
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            track_ids = (
                result.boxes.id
                .detach()
                .cpu()
                .numpy()
                .astype(np.int64)
            )

            for raw_box, det_conf, track_id in zip(
                raw_boxes,
                raw_confs,
                track_ids,
            ):
                track_id = int(track_id)

                unique_ids.add(
                    track_id
                )

                if track_id not in track_states:
                    track_states[
                        track_id
                    ] = create_track_state(
                        track_id,
                        raw_box,
                        source_frame_idx,
                        float(det_conf),
                    )

                state = track_states[
                    track_id
                ]

                update_smoothed_box(
                    state,
                    raw_box,
                )

                state["last_raw_box"] = (
                    np.asarray(
                        raw_box,
                        dtype=np.float32,
                    ).copy()
                )

                state["last_det_conf"] = float(
                    det_conf
                )

                state["last_seen_frame"] = int(
                    source_frame_idx
                )

                state["missed_frames"] = 0
                state["hits"] += 1

                state["trail"].append(
                    state_center(
                        state["smooth_box"]
                    )
                )

                current_tracks.append({
                    "track_id": track_id,
                    "state": state,
                    "raw_box": (
                        np.asarray(
                            raw_box,
                            dtype=np.float32,
                        ).copy()
                    ),
                })

        current_ids = {
            x["track_id"]
            for x in current_tracks
        }

        # ----------------------------------------------------
        # 2) Crop from smoothed boxes
        # ----------------------------------------------------

        crops = []
        crop_track_ids = []

        for item in current_tracks:
            track_id = item[
                "track_id"
            ]

            state = item[
                "state"
            ]

            crop, _ = padded_crop_from_xyxy(
                frame,
                state["smooth_box"],
                padding=CONTEXT_PAD,
            )

            if crop is None:
                continue

            crops.append(crop)
            crop_track_ids.append(track_id)

        # ----------------------------------------------------
        # 3) Classifier + temporal class smoothing
        # ----------------------------------------------------

        classifier_start = time.perf_counter()

        current_prob_vectors = classify_crops(
            crops
        )

        classifier_ms = (
            time.perf_counter()
            - classifier_start
        ) * 1000.0

        raw_cls_by_id = {}

        for track_id, current_probs in zip(
            crop_track_ids,
            current_prob_vectors,
        ):
            state = track_states[
                track_id
            ]

            raw_idx = int(
                np.argmax(current_probs)
            )

            raw_species = idx_to_class[
                raw_idx
            ]

            raw_cls_conf = float(
                current_probs[raw_idx]
            )

            if (
                state["last_raw_species"] is not None
                and raw_species != state["last_raw_species"]
            ):
                state["raw_species_switches"] += 1

            state["last_raw_species"] = raw_species

            state["prob_ema"] = ema_probs(
                state["prob_ema"],
                current_probs,
                CLASS_PROB_EMA_ALPHA,
            )

            stable_idx = int(
                np.argmax(
                    state["prob_ema"]
                )
            )

            stable_species = idx_to_class[
                stable_idx
            ]

            stable_conf = float(
                state["prob_ema"][
                    stable_idx
                ]
            )

            if (
                state["last_stable_species"] is not None
                and stable_species != state["last_stable_species"]
            ):
                state["stable_species_switches"] += 1

            state["last_stable_species"] = stable_species
            state["species"] = stable_species
            state["cls_conf"] = stable_conf

            raw_cls_by_id[
                track_id
            ] = {
                "raw_species": raw_species,
                "raw_cls_conf": raw_cls_conf,
            }

        # ----------------------------------------------------
        # 4) Short-gap hold
        # ----------------------------------------------------

        held_tracks = []

        for track_id, state in list(
            track_states.items()
        ):
            if track_id in current_ids:
                continue

            age = (
                source_frame_idx
                - state["last_seen_frame"]
            )

            state["missed_frames"] = int(
                age
            )

            if (
                DRAW_HELD_TRACKS
                and state["hits"] >= MIN_TRACK_HITS_TO_DRAW
                and 1 <= age <= MAX_MISSED_FRAMES
            ):
                held_tracks.append({
                    "track_id": track_id,
                    "state": state,
                    "display_box": predicted_hold_box(
                        state,
                        age,
                    ),
                })

            if age > STATE_MAX_AGE:
                del track_states[
                    track_id
                ]

        frame_rows = []

        # ----------------------------------------------------
        # 5) Draw detected tracks
        # ----------------------------------------------------

        for item in current_tracks:
            track_id = item[
                "track_id"
            ]

            state = track_states[
                track_id
            ]

            if (
                state["hits"]
                < MIN_TRACK_HITS_TO_DRAW
            ):
                continue

            draw_track(
                frame,
                state,
                source="DETECTED",
                display_box=state["smooth_box"],
            )

            detected_track_rows += 1

            raw_box = item[
                "raw_box"
            ]

            display_box = state[
                "smooth_box"
            ]

            raw_cls = raw_cls_by_id.get(
                track_id,
                {
                    "raw_species": None,
                    "raw_cls_conf": np.nan,
                },
            )

            frame_rows.append({
                "video": video_path.name,
                "source_frame": source_frame_idx,
                "output_frame": output_frame_idx,
                "time_sec": source_frame_idx / source_fps,

                "track_id": track_id,
                "source": "DETECTED",

                "raw_x1": float(raw_box[0]),
                "raw_y1": float(raw_box[1]),
                "raw_x2": float(raw_box[2]),
                "raw_y2": float(raw_box[3]),

                "display_x1": float(display_box[0]),
                "display_y1": float(display_box[1]),
                "display_x2": float(display_box[2]),
                "display_y2": float(display_box[3]),

                "det_conf": float(
                    state["last_det_conf"]
                ),

                "raw_species": raw_cls[
                    "raw_species"
                ],

                "raw_cls_conf": raw_cls[
                    "raw_cls_conf"
                ],

                "stable_species": state[
                    "species"
                ],

                "stable_cls_conf": float(
                    state["cls_conf"]
                ),

                "missed_frames": 0,
                "track_hits": int(
                    state["hits"]
                ),

                "tracker_detector_ms": tracker_ms,
                "classifier_ms": classifier_ms,
                "frame_pipeline_ms": np.nan,
            })

        # ----------------------------------------------------
        # 6) Draw held tracks
        # ----------------------------------------------------

        for item in held_tracks:
            track_id = item[
                "track_id"
            ]

            state = item[
                "state"
            ]

            display_box = item[
                "display_box"
            ]

            draw_track(
                frame,
                state,
                source="HELD",
                display_box=display_box,
            )

            held_track_rows += 1

            raw_box = state[
                "last_raw_box"
            ]

            frame_rows.append({
                "video": video_path.name,
                "source_frame": source_frame_idx,
                "output_frame": output_frame_idx,
                "time_sec": source_frame_idx / source_fps,

                "track_id": track_id,
                "source": "HELD",

                "raw_x1": float(raw_box[0]),
                "raw_y1": float(raw_box[1]),
                "raw_x2": float(raw_box[2]),
                "raw_y2": float(raw_box[3]),

                "display_x1": float(display_box[0]),
                "display_y1": float(display_box[1]),
                "display_x2": float(display_box[2]),
                "display_y2": float(display_box[3]),

                "det_conf": float(
                    state["last_det_conf"]
                ),

                "raw_species": None,
                "raw_cls_conf": np.nan,

                "stable_species": state[
                    "species"
                ],

                "stable_cls_conf": float(
                    state["cls_conf"]
                ),

                "missed_frames": int(
                    state["missed_frames"]
                ),

                "track_hits": int(
                    state["hits"]
                ),

                "tracker_detector_ms": tracker_ms,
                "classifier_ms": 0.0,
                "frame_pipeline_ms": np.nan,
            })

        # ----------------------------------------------------
        # 7) Write frame + rows
        # ----------------------------------------------------

        writer.write(frame)

        pipeline_ms = (
            time.perf_counter()
            - frame_start
        ) * 1000.0

        for row in frame_rows:
            row["frame_pipeline_ms"] = pipeline_ms
            csv_writer.writerow(row)

        tracker_times.append(tracker_ms)
        classifier_times.append(classifier_ms)
        pipeline_times.append(pipeline_ms)

        output_frame_idx += 1

        if (
            output_frame_idx % 100
            == 0
        ):
            recent_ms = float(
                np.mean(
                    pipeline_times[-100:]
                )
            )

            recent_fps = (
                1000.0 / recent_ms
                if recent_ms > 0
                else 0.0
            )

            print(
                f"Frames={output_frame_idx}"
                f" | IDs={len(unique_ids)}"
                f" | active={len(track_states)}"
                f" | held={held_track_rows}"
                f" | recent={recent_ms:.1f} ms"
                f" (~{recent_fps:.2f} FPS)"
            )

        del result

        if output_frame_idx % 50 == 0:
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    cap.release()
    writer.release()
    csv_file.close()

    video_wall_sec = (
        time.perf_counter()
        - video_start
    )

    mean_tracker_ms = (
        float(np.mean(tracker_times))
        if tracker_times
        else 0.0
    )

    mean_classifier_ms = (
        float(np.mean(classifier_times))
        if classifier_times
        else 0.0
    )

    mean_pipeline_ms = (
        float(np.mean(pipeline_times))
        if pipeline_times
        else 0.0
    )

    pipeline_fps = (
        1000.0 / mean_pipeline_ms
        if mean_pipeline_ms > 0
        else 0.0
    )

    raw_switches = int(
        sum(
            s["raw_species_switches"]
            for s in track_states.values()
        )
    )

    stable_switches = int(
        sum(
            s["stable_species_switches"]
            for s in track_states.values()
        )
    )

    summary = {
        "video": video_path.name,
        "width": W,
        "height": H,
        "source_fps": source_fps,
        "source_frames_reported": reported_frames,
        "processed_frames": output_frame_idx,

        "unique_track_ids": len(unique_ids),
        "detected_track_rows": detected_track_rows,
        "held_track_rows": held_track_rows,

        "mean_tracker_detector_ms": mean_tracker_ms,
        "mean_classifier_ms": mean_classifier_ms,
        "mean_pipeline_ms": mean_pipeline_ms,
        "pipeline_fps": pipeline_fps,

        "video_wall_sec": video_wall_sec,

        "raw_species_switches_remaining_states": raw_switches,
        "stable_species_switches_remaining_states": stable_switches,

        "annotated_video": str(local_video),
        "tracks_csv": str(local_csv),
    }

    all_video_summaries.append(
        summary
    )

    drive_video_dir = (
        OUTPUT_DRIVE_ROOT
        / safe_stem
    )

    if drive_video_dir.exists():
        shutil.rmtree(
            drive_video_dir
        )

    shutil.copytree(
        local_dir,
        drive_video_dir,
    )

    summary_path = (
        drive_video_dir
        / f"{safe_stem}_V2_summary.json"
    )

    with open(
        summary_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            summary,
            f,
            indent=2,
            ensure_ascii=False,
        )

    print("\nCompleted:", video_path.name)
    print("Frames          :", output_frame_idx)
    print("Unique Track IDs:", len(unique_ids))
    print("Held rows       :", held_track_rows)
    print(
        "Mean pipeline   :",
        f"{mean_pipeline_ms:.2f} ms/frame"
    )
    print(
        "Approx FPS      :",
        f"{pipeline_fps:.2f}"
    )
    print("Saved:", drive_video_dir)

    del video_detector
    del track_states

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\nSTATUS: ALL_V2_VIDEOS_PROCESSED")


In [ ]:
# ============================================================
# CELL 9 — FINAL SUMMARY + ZIP OUTPUTS
# ============================================================

summary_df = pd.DataFrame(
    all_video_summaries
)

SUMMARY_CSV = (
    OUTPUT_DRIVE_ROOT
    / "all_video_V2_summary.csv"
)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)

GLOBAL_JSON = (
    OUTPUT_DRIVE_ROOT
    / "video_V2_experiment_summary.json"
)

global_summary = {
    "experiment": (
        "Stage3_YOLO11n_BoTSORT_BBoxEMA_"
        "MobileNetV4V3_ClassEMA_VideoV2"
    ),

    "detector_checkpoint": str(
        DETECTOR_CKPT
    ),

    "classifier_checkpoint": str(
        CLASSIFIER_CKPT
    ),

    "tracker": "BoT-SORT",
    "tracker_yaml": str(TRACKER_YAML),

    "detector_imgsz": DETECTOR_IMGSZ,

    "track_input_conf": TRACK_INPUT_CONF,
    "track_high_thresh": TRACK_HIGH_THRESH,
    "track_low_thresh": TRACK_LOW_THRESH,
    "new_track_thresh": NEW_TRACK_THRESH,
    "track_buffer": TRACK_BUFFER,
    "match_thresh": TRACK_MATCH_THRESH,

    "bbox_ema_alpha": BBOX_EMA_ALPHA,
    "velocity_ema_alpha": VELOCITY_EMA_ALPHA,

    "max_missed_frames": MAX_MISSED_FRAMES,
    "min_track_hits_to_draw": MIN_TRACK_HITS_TO_DRAW,

    "classifier_input": CLASSIFIER_INPUT,
    "context_padding": CONTEXT_PAD,
    "class_probability_ema_alpha": CLASS_PROB_EMA_ALPHA,

    "overlay": {
        "box_thickness": BOX_THICKNESS,
        "held_box_thickness": HELD_BOX_THICKNESS,
        "font_scale": FONT_SCALE,
        "text_thickness": TEXT_THICKNESS,
    },

    "videos_processed": int(
        len(summary_df)
    ),

    "total_frames_processed": int(
        summary_df["processed_frames"].sum()
        if len(summary_df)
        else 0
    ),

    "total_unique_track_ids_sum": int(
        summary_df["unique_track_ids"].sum()
        if len(summary_df)
        else 0
    ),

    "total_held_track_rows": int(
        summary_df["held_track_rows"].sum()
        if len(summary_df)
        else 0
    ),

    "mean_pipeline_fps_unweighted": float(
        summary_df["pipeline_fps"].mean()
        if len(summary_df)
        else 0.0
    ),

    "training_performed": False,
}

with open(
    GLOBAL_JSON,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        global_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )

ZIP_BASE = Path(
    "/content/Coral_Video_V2_Stable_Outputs"
)

zip_path = Path(
    shutil.make_archive(
        str(ZIP_BASE),
        "zip",
        root_dir=str(
            OUTPUT_DRIVE_ROOT
        ),
    )
)

DRIVE_ZIP = (
    OUTPUT_DRIVE_ROOT
    / zip_path.name
)

shutil.copy2(
    zip_path,
    DRIVE_ZIP,
)

print("=" * 125)
print("VIDEO V2 COMPLETE")
print("=" * 125)

if len(summary_df):
    cols = [
        "video",
        "processed_frames",
        "unique_track_ids",
        "detected_track_rows",
        "held_track_rows",
        "mean_tracker_detector_ms",
        "mean_classifier_ms",
        "mean_pipeline_ms",
        "pipeline_fps",
    ]

    print(
        summary_df[
            cols
        ].to_string(
            index=False
        )
    )

print("\nOutput root:")
print(OUTPUT_DRIVE_ROOT)

print("\nSummary CSV:")
print(SUMMARY_CSV)

print("\nSummary JSON:")
print(GLOBAL_JSON)

print("\nAll outputs ZIP:")
print(DRIVE_ZIP)

print("\nSTATUS: VIDEO_V2_COMPLETE")


In [ ]:
# ============================================================
# CELL 10 — OPTIONAL PREVIEW
# ============================================================

from IPython.display import Video, display

annotated = sorted(
    OUTPUT_DRIVE_ROOT.rglob(
        "*_V2_tracked_smoothed.mp4"
    )
)

if not annotated:
    print("No annotated V2 video found.")

else:
    preview = annotated[0]

    print("Preview:")
    print(preview)

    display(
        Video(
            str(preview),
            embed=True,
            width=950,
        )
    )


## Quick tuning

If the box is still jittery:

```python
BBOX_EMA_ALPHA = 0.35
```

If it becomes too slow/laggy:

```python
BBOX_EMA_ALPHA = 0.55
```

For even larger overlay:

```python
BOX_THICKNESS = 6
FONT_SCALE = 1.0
TEXT_THICKNESS = 3
```

If held boxes create ghosts:

```python
MAX_MISSED_FRAMES = 1
```

When Stage 3.1 or Stage 4.1 wins, change only `DETECTOR_CKPT`.
